# CBX8 LD matrix from 1000 Genomes (GRCh38)

This notebook reads `output/CBX8_variants.vcf`, downloads the 1000G sample panel, 
pulls only the chr17 region spanning those variants from the 1000 Genomes GRCh38 VCF, 
and builds an LD correlation matrix (R) for EUR samples.

Outputs:
- `output/CBX8_LD_R.csv` (LD matrix)
- `output/CBX8_LD_R.npy` (binary matrix)
- `output/CBX8_LD_variant_order.tsv` (variant order)


In [ ]:
import importlib.util
import subprocess
import sys

def ensure_pkg(name):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', name])

for pkg in ['numpy', 'pysam']:
    ensure_pkg(pkg)


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / 'utils').is_dir()
            and (candidate / 'vignettes').is_dir()
            and (candidate / 'README.md').exists()
        ):
            return candidate
    raise FileNotFoundError('Could not locate eQTL_annotations_for_susine project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.paths import ensure_project_dirs

PATHS = ensure_project_dirs(PROJECT_ROOT)
data_dir = PATHS.data
z_score_output_dir = PATHS.output_z_score
ld_output_dir = PATHS.output_ld
prelim_output_dir = PATHS.output_prelim

vcf_path = z_score_output_dir / 'CBX8_variants.vcf'
if not vcf_path.exists():
    raise FileNotFoundError(f'Missing input VCF: {vcf_path}')

variants = []
with vcf_path.open() as f:
    for line in f:
        if line.startswith('#'):
            continue
        fields = line.rstrip('\n').split('\t')
        if len(fields) < 5:
            continue
        chrom, pos, vid, ref, alt = fields[:5]
        variants.append({
            'chrom': chrom,
            'pos': int(pos),
            'id': vid,
            'ref': ref,
            'alt': alt.split(',')[0],
        })

if not variants:
    raise ValueError('No variants found in input VCF.')

chroms = sorted({v['chrom'] for v in variants})
min_pos = min(v['pos'] for v in variants)
max_pos = max(v['pos'] for v in variants)

print(f'Project root: {PROJECT_ROOT}')
print(f'Loaded {len(variants)} variants on {chroms}')
print(f'Region: {chroms[0]}:{min_pos}-{max_pos}')


In [ ]:
import urllib.request

panel_url = (
    'https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/'
    'integrated_call_samples_v3.20130502.ALL.panel'
)
panel_path = data_dir / 'integrated_call_samples_v3.20130502.ALL.panel'

if not panel_path.exists():
    print(f'Downloading panel to {panel_path}')
    urllib.request.urlretrieve(panel_url, panel_path)

eur_samples = []
with panel_path.open() as f:
    header = f.readline()
    for line in f:
        line = line.strip()
        if not line:
            continue
        fields = line.split()
        if len(fields) < 3:
            continue
        sample, pop, super_pop = fields[:3]
        if super_pop == 'EUR':
            eur_samples.append(sample)

print(f'EUR samples in panel: {len(eur_samples)}')


In [ ]:
import numpy as np
import pysam

base_url = (
    'https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/'
    'data_collections/1000_genomes_project/release/'
    '20190312_biallelic_SNV_and_INDEL'
)

chrom_no_chr = chroms[0].replace('chr', '')
vcf_url = (
    f'{base_url}/ALL.chr{chrom_no_chr}.'
    'shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz'
)

vcf_in = pysam.VariantFile(vcf_url)
contigs = set(vcf_in.header.contigs)
chrom_with_chr = f'chr{chrom_no_chr}'

if chroms[0] in contigs:
    fetch_chrom = chroms[0]
elif chrom_with_chr in contigs:
    fetch_chrom = chrom_with_chr
elif chrom_no_chr in contigs:
    fetch_chrom = chrom_no_chr
else:
    raise ValueError(f'Chromosome not found in VCF contigs: {chroms[0]}')

vcf_samples = list(vcf_in.header.samples)
eur_set = set(eur_samples)
sample_names = [s for s in vcf_samples if s in eur_set]
if not sample_names:
    raise ValueError('No EUR samples from the panel were found in the VCF header.')

print(f'EUR samples used: {len(sample_names)}')

key_to_index = {}
for idx, v in enumerate(variants):
    chrom_key = v['chrom']
    chrom_no_chr = chrom_key.replace('chr', '')
    chrom_with_chr = f'chr{chrom_no_chr}'
    for ck in {chrom_key, chrom_no_chr, chrom_with_chr}:
        key_to_index[(ck, v['pos'], v['ref'], v['alt'])] = idx

G = np.full((len(variants), len(sample_names)), np.nan, dtype=np.float32)

start0 = min_pos - 1
end0 = max_pos
hits = 0
for rec in vcf_in.fetch(fetch_chrom, start0, end0):
    if not rec.alts:
        continue
    alt = rec.alts[0]
    key = (rec.chrom, rec.pos, rec.ref, alt)
    idx = key_to_index.get(key)
    if idx is None:
        continue
    hits += 1
    for j, s in enumerate(sample_names):
        gt = rec.samples[s].get('GT')
        if gt is None or None in gt:
            continue
        if -1 in gt:
            continue
        G[idx, j] = gt[0] + gt[1]

print(f'Matched variants in 1000G: {hits} / {len(variants)}')

found_mask = ~np.isnan(G).all(axis=1)
if not found_mask.all():
    missing = [variants[i]['id'] for i, ok in enumerate(found_mask) if not ok]
    print(f'Missing variants not found in 1000G VCF: {len(missing)}')
    print('Example missing IDs:', missing[:10])
    G = G[found_mask]
    variants = [v for v, ok in zip(variants, found_mask) if ok]

try:
    vcf_in.close()
except OSError as exc:
    print(f'Warning: non-fatal error while closing remote 1000G VCF: {exc}')


In [ ]:
import csv

import pandas as pd

LD_DECIMALS = 3
WRITE_DENSE_LD_ARTIFACTS = False

G_mean = np.nanmean(G, axis=1, keepdims=True)
G_filled = np.where(np.isnan(G), G_mean, G)
X = G_filled - G_mean
ddof = 1 if G.shape[1] > 1 else 0
stds = X.std(axis=1, ddof=ddof, keepdims=True)
stds[stds == 0] = np.nan
X = X / stds
X = np.nan_to_num(X, nan=0.0)
R = (X @ X.T) / max(G.shape[1] - 1, 1)
np.fill_diagonal(R, 1.0)
R = np.round(R, LD_DECIMALS).astype(np.float32)

variant_labels = [v['id'] for v in variants]

output_csv = ld_output_dir / 'CBX8_LD_R.csv'
output_npy = ld_output_dir / 'CBX8_LD_R.npy'
if WRITE_DENSE_LD_ARTIFACTS:
    with output_csv.open('w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['variant'] + variant_labels)
        for label, row in zip(variant_labels, R):
            writer.writerow([label] + [f'{x:.{LD_DECIMALS}f}' for x in row])
    np.save(output_npy, R)
    print('Wrote:', output_csv)
    print('Wrote:', output_npy)
else:
    print('Skipping dense LD artifacts (CSV/NPY); using compact Parquet bundle instead.')

order_path = ld_output_dir / 'CBX8_LD_variant_order.tsv'
with order_path.open('w', newline='') as f:
    writer = csv.writer(f, delimiter='	')
    writer.writerow(['index', 'id', 'chrom', 'pos', 'ref', 'alt'])
    for i, v in enumerate(variants):
        writer.writerow([i, v['id'], v['chrom'], v['pos'], v['ref'], v['alt']])

print('Wrote:', order_path)

z_scores_path = z_score_output_dir / 'CBX8_GTEx_z_scores.csv'
if not z_scores_path.exists():
    raise FileNotFoundError(f'Missing z-score export: {z_scores_path}')

z_scores = pd.read_csv(z_scores_path)
ld_index = {variant_id: idx for idx, variant_id in enumerate(variant_labels)}
master_df = z_scores[z_scores['variant_id'].isin(ld_index)].copy()
master_df['ld_included'] = True
master_df['ld_matrix_index'] = master_df['variant_id'].map(ld_index)
master_df = master_df.sort_values('ld_matrix_index').reset_index(drop=True)

if len(master_df) != len(variant_labels):
    missing_from_z = [variant_id for variant_id in variant_labels if variant_id not in set(master_df['variant_id'])]
    raise ValueError(f'LD matched variants missing from z-score table: {missing_from_z[:10]}')

master_path = ld_output_dir / 'CBX8_phase1_master_variants.csv'
master_df.to_csv(master_path, index=False)
print('Wrote:', master_path)

variant_map_df = pd.DataFrame(
    {
        'snp_index': np.arange(len(variants), dtype=np.int32),
        'variant_id': [v['id'] for v in variants],
        'chrom': [v['chrom'] for v in variants],
        'pos': [v['pos'] for v in variants],
        'ref': [v['ref'] for v in variants],
        'alt': [v['alt'] for v in variants],
    }
)
variant_map_path = ld_output_dir / 'CBX8_phase1_variant_map.parquet'
variant_map_df.to_parquet(variant_map_path, index=False, compression='zstd')
print('Wrote:', variant_map_path)

z_compact_df = master_df[['variant_id', 'z_score', 'sample_size']].merge(
    variant_map_df[['snp_index', 'variant_id']],
    on='variant_id',
    how='inner',
).sort_values('snp_index').reset_index(drop=True)
z_compact_path = ld_output_dir / 'CBX8_phase1_z_scores.parquet'
z_compact_df.to_parquet(z_compact_path, index=False, compression='zstd')
print('Wrote:', z_compact_path)

tri_i, tri_j = np.triu_indices(len(variant_labels), k=1)
ld_long_df = pd.DataFrame(
    {
        'snp_index_1': tri_i.astype(np.int32),
        'snp_index_2': tri_j.astype(np.int32),
        'r': R[tri_i, tri_j].astype(np.float32),
    }
)
ld_long_path = ld_output_dir / 'CBX8_phase1_LD_R_long.parquet'
ld_long_df.to_parquet(ld_long_path, index=False, compression='zstd')
print('Wrote:', ld_long_path)

z = master_df['z_score'].to_numpy(dtype=np.float64)
A = np.abs(R.astype(np.float64, copy=False))
ut_mask = np.triu(np.ones(A.shape, dtype=bool), k=1)
ut_abs = A[ut_mask]
R_sq = np.square(R.astype(np.float64, copy=False))
ut_sq = R_sq[ut_mask]
z2 = np.square(z)
z4 = np.square(z2)

metrics_df = pd.DataFrame(
    [
        {
            'gene_name': 'CBX8',
            'n_variants_z': len(z_scores),
            'n_variants_ld': len(variant_labels),
            'n_variants_master': len(master_df),
            'M1': float(2.0 * np.mean(ut_abs * (1.0 - ut_abs))),
            'M2': float(2.0 * np.mean(ut_sq * (1.0 - ut_sq))),
            'z_count_abs_gt_3': int(np.sum(np.abs(z) > 3.0)),
            'z_eff_signals': float((np.sum(z2) ** 2) / np.sum(z4)) if np.sum(z4) > 0 else np.nan,
        }
    ]
)
metrics_path = prelim_output_dir / 'CBX8_phase1_dataset_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print('Wrote:', metrics_path)
print(metrics_df.to_string(index=False))

funnel_path = prelim_output_dir / 'CBX8_phase1_count_funnel.csv'
if funnel_path.exists():
    count_funnel = pd.read_csv(funnel_path)
else:
    count_funnel = pd.DataFrame(columns=['step', 'count'])

ld_steps = pd.DataFrame(
    [
        {'step': 'ld_matched', 'count': len(variant_labels)},
        {'step': 'ld_unmatched', 'count': len(z_scores) - len(variant_labels)},
        {'step': 'phase1_master_z_intersect_ld', 'count': len(master_df)},
    ]
)
count_funnel = pd.concat(
    [count_funnel[~count_funnel['step'].isin(ld_steps['step'])], ld_steps],
    ignore_index=True,
)
count_funnel.to_csv(funnel_path, index=False)
print('Wrote:', funnel_path)
print(count_funnel)
